![MuJoCo banner](https://raw.githubusercontent.com/google-deepmind/mujoco/main/banner.png)







### Copyright notice

> <p><small><small>Copyright 2025 DeepMind Technologies Limited.</small></p>
> <p><small><small>Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at <a href="http://www.apache.org/licenses/LICENSE-2.0">http://www.apache.org/licenses/LICENSE-2.0</a>.</small></small></p>
> <p><small><small>Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.</small></small></p>

# Vision in The Playground! <a href="https://colab.research.google.com/github/google-deepmind/mujoco_playground/blob/main/learning/notebooks/vision.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" width="140" align="center"/></a>

In this notebook, we'll walk through two vision environments available in MuJoCo Playground that use the MJWarp Batch Renderer.

**A Colab runtime with GPU acceleration is required.** If you're using a CPU-only runtime, you can switch using the menu "Runtime > Change runtime type".

In [ ]:
#@title Install pre-requisites
!pip install warp-lang
!pip install mujoco
!pip install mujoco_mjx
!pip install brax

In [ ]:
# @title Check if MuJoCo installation was successful

import distutils.util
import os
import subprocess

if subprocess.run('nvidia-smi').returncode:
  raise RuntimeError(
      'Cannot communicate with GPU. '
      'Make sure you are using a GPU Colab runtime. '
      'Go to the Runtime menu and select Choose runtime type.'
  )

# Add an ICD config so that glvnd can pick up the Nvidia EGL driver.
# This is usually installed as part of an Nvidia driver package, but the Colab
# kernel doesn't install its driver via APT, and as a result the ICD is missing.
# (https://github.com/NVIDIA/libglvnd/blob/master/src/EGL/icd_enumeration.md)
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")

# Configure MuJoCo to use the EGL rendering backend (requires GPU)
print('Setting environment variable to use GPU rendering:')
%env MUJOCO_GL=egl

try:
  print('Checking that the installation succeeded:')
  import mujoco

  mujoco.MjModel.from_xml_string('<mujoco/>')
except Exception as e:
  raise e from RuntimeError(
      'Something went wrong during installation. Check the shell output above '
      'for more information.\n'
      'If using a hosted Colab runtime, make sure you enable GPU acceleration '
      'by going to the Runtime menu and selecting "Choose runtime type".'
  )

print('Installation successful.')

# Tell XLA to use Triton GEMM, this improves steps/sec by ~30% on some GPUs
xla_flags = os.environ.get('XLA_FLAGS', '')
xla_flags += ' --xla_gpu_triton_gemm_any=True'
os.environ['XLA_FLAGS'] = xla_flags

In [ ]:
# @title Import packages for plotting and creating graphics
import json
import itertools
import time
from typing import Callable, List, NamedTuple, Optional, Union
import numpy as np

# Graphics and plotting.
print("Installing mediapy:")
!command -v ffmpeg >/dev/null || (apt update && apt install -y ffmpeg)
!pip install -q mediapy
import mediapy as media
import matplotlib.pyplot as plt

# More legible printing from numpy.
np.set_printoptions(precision=3, suppress=True, linewidth=100)

In [ ]:
# @title Import MuJoCo, MJX, and Brax
from datetime import datetime
import functools
import os
from typing import Any, Dict, Sequence, Tuple, Union
from brax import base
from brax import envs
from brax import math
from brax.base import Base, Motion, Transform
from brax.base import State as PipelineState
from brax.envs.base import Env, PipelineEnv, State
from brax.io import html, mjcf, model
from brax.mjx.base import State as MjxState
from custombrax.training.agents.ppo import networks_vision as ppo_networks_vision
from custombrax.training.agents.ppo import train as ppo
from custombrax.training.agents.sac import networks as sac_networks
from custombrax.training.agents.sac import train as sac
from etils import epath
from flax import struct
from flax.training import orbax_utils
from IPython.display import HTML, clear_output
import jax
from jax import numpy as jp
from matplotlib import pyplot as plt
import mediapy as media
from ml_collections import config_dict
import mujoco
from mujoco import mjx
import numpy as np
from orbax import checkpoint as ocp

In [ ]:
#@title Install MuJoCo Playground
!pip install playground

In [ ]:
#@title Import The Playground

from mujoco_playground import wrapper
from mujoco_playground import registry

# Vision

MuJoCo Playground includes two vision environments that use the MJWarp Batch Renderer. The first is `CartpoleBalance`, and the second is `FrankaPickCubeCartesian`.

# CartpoleBalance

Let's start with the simple `CartpoleBalance` environment. The goal is to balance the pole using only image input from a fixed front camera. 

When loading vision environments, we need to specify that `vision` is `True` in the environment config, and we need to tell the vision config how many environment we plan to render. The second is important as the renderer is initialized in advance to render a specific number of worlds.Let's load the environment:

In [ ]:
env_name = "CartpoleBalance"
env_cfg = registry.get_default_config("CartpoleBalance")
env_cfg["vision"] = True
env = registry.load("CartpoleBalance", config_overrides=env_cfg)

In [ ]:
env_cfg

## Train Policy

Let's train the vision policy and visualize the rollouts. The policy takes roughly 7 minutes to train on a RTX4090 GPU.

To train vision policies, use `brax_vision_ppo_config` instead of `brax_ppo_config`. Similarly, our network factory uses brax's `make_ppo_networks_vision` instead of `make_ppo_networks`.

In [ ]:
from mujoco_playground.config import dm_control_suite_params
ppo_params = dm_control_suite_params.brax_vision_ppo_config(env_name)
ppo_params

### PPO

In [ ]:
x_data, y_data, y_dataerr = [], [], []
times = [datetime.now()]


def progress(num_steps, metrics):
  clear_output(wait=True)

  times.append(datetime.now())
  x_data.append(num_steps)
  y_data.append(metrics["eval/episode_reward"])
  y_dataerr.append(metrics["eval/episode_reward_std"])

  plt.xlim([0, ppo_params["num_timesteps"] * 1.25])
  plt.ylim([0, 30])
  plt.xlabel("# environment steps")
  plt.ylabel("reward per episode")
  plt.title(f"y={y_data[-1]:.3f}")
  plt.errorbar(x_data, y_data, yerr=y_dataerr, color="blue")

  display(plt.gcf())

ppo_training_params = dict(ppo_params)
network_factory = ppo_networks_vision.make_ppo_networks_vision
if "network_factory" in ppo_params:
  del ppo_training_params["network_factory"]
  network_factory = functools.partial(
      ppo_networks_vision.make_ppo_networks_vision,
      **ppo_params.network_factory
  )

num_eval_envs = ppo_params.get("num_eval_envs", 128)

if "num_eval_envs" in ppo_training_params:
  del ppo_training_params["num_eval_envs"]

train_fn = functools.partial(
    ppo.train,
    **dict(ppo_training_params),
    network_factory=network_factory,
    progress_fn=progress,
    seed=1,
    num_eval_envs=num_eval_envs
)

In [ ]:
eval_env_cfg = config_dict.ConfigDict(env_cfg)
eval_env_cfg.vision_config.nworld = num_eval_envs
eval_env = registry.load(env_name, config=eval_env_cfg)

make_inference_fn, params, metrics = train_fn(
    environment=env,
    wrap_env_fn=wrapper.wrap_for_brax_training,
    eval_env=eval_env,
)
print(f"time to jit: {times[1] - times[0]}")
print(f"time to train: {times[-1] - times[1]}")

## Visualize Rollouts

In [ ]:
infer_env_cfg = config_dict.ConfigDict(env_cfg)
infer_env_cfg.vision_config.nworld = 1
infer_env = registry.load(env_name, config=infer_env_cfg)
wrapped_infer_env = wrapper.wrap_for_brax_training(
    infer_env,
    episode_length=ppo_params.episode_length,
    action_repeat=ppo_params.get("action_repeat", 1),
)

jit_reset = jax.jit(wrapped_infer_env.reset)
jit_step = jax.jit(wrapped_infer_env.step)
jit_inference_fn = jax.jit(make_inference_fn(params, deterministic=True))

In [ ]:
rng = jax.random.split(jax.random.PRNGKey(42), 1)
reset_states = jax.jit(wrapped_infer_env.reset)(rng)

empty_data = reset_states.data.__class__(
    **{k: None for k in reset_states.data.__annotations__}
)  # pytype: disable=attribute-error
empty_traj = reset_states.__class__(
    **{k: None for k in reset_states.__annotations__}
)  # pytype: disable=attribute-error
empty_traj = empty_traj.replace(data=empty_data)

def step(carry, _):
  state, rng = carry
  rng, act_key = jax.random.split(rng)
  act_keys = jax.random.split(act_key, 1)
  act = jax.vmap(jit_inference_fn)(state.obs, act_keys)[0]
  state = wrapped_infer_env.step(state, act)
  traj_data = empty_traj.tree_replace({
      "data.qpos": state.data.qpos,
      "data.qvel": state.data.qvel,
      "data.time": state.data.time,
      "data.ctrl": state.data.ctrl,
      "data.mocap_pos": state.data.mocap_pos,
      "data.mocap_quat": state.data.mocap_quat,
      "data.xfrc_applied": state.data.xfrc_applied,
  })
  return (state, rng), traj_data

@jax.jit
def do_rollout(state, rng):
  _, traj = jax.lax.scan(
      step, (state, rng), None, length=ppo_params.episode_length
  )
  return traj

traj_stacked = do_rollout(reset_states, jax.random.PRNGKey(42 + 1))
# traj_stacked has shape (time, nworld, ...), swap to (nworld, time, ...).
traj_stacked = jax.tree.map(lambda x: jp.moveaxis(x, 0, 1), traj_stacked)

# Use only world 0 and render a single video.
traj_world0 = jax.tree.map(lambda x: x[0], traj_stacked)
rollout = [
    jax.tree.map(lambda x, j=j: x[j], traj_world0)
    for j in range(ppo_params.episode_length)
]

render_every = 2
fps = 1.0 / infer_env.dt / render_every
print(f"FPS for rendering: {fps}")
scene_option = mujoco.MjvOption()
scene_option.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT] = False
scene_option.flags[mujoco.mjtVisFlag.mjVIS_PERTFORCE] = False
scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTFORCE] = False

traj = rollout[::render_every]
frames = infer_env.render(
    traj, height=480, width=640, scene_option=scene_option
)
media.show_video(frames, fps=fps)

# Vision Manipulation

Let's now train vision manipulation using a pixels-to-action policy with the Franka Emika Panda robot. To keep the runtime for this task short and to simplify real-world transfer, we've adapted the [bring_to_target](https://github.com/kevinzakka/mujoco_playground/blob/main/mujoco_playground/_src/manipulation/franka_emika_panda/bring_to_target.py) task to the 2D case and introduce a task-space controller. These changes allow the policy to converge in well under ten million samples using a low-resolution single-camera RGB input. Collaborators from the [MedCVR](https://medcvr.utm.utoronto.ca/) lab have used this code for [real-world deployment](https://www.youtube.com/watch?v=Y_lHJDttlHY).

*Domain Randomization*

Just like in physics simulation, the mismatch between the simulated environment that the policy is trained on and the real-world setting in which it is deployed can be minimised either by meticulously calibrating the simulation or by randomizing simulation parameters such that the policy learns to be robust to a wide variety of ranges. The MJWarp Batch Renderer supports domain randomisation, rendering different camera poses and geometry appearances across parallel environments.



In [ ]:
env_name = "PandaPickCubeCartesian"
env_cfg = registry.get_default_config(env_name)
env_cfg["vision"] = True
env = registry.load(env_name, config=env_cfg)

# Train Policy

Let's train our manipulation policy and visualize the rollouts. The policy takes roughly 7 minutes to train on a RTX 4090 GPU.

In [ ]:
from mujoco_playground.config import manipulation_params
ppo_params = manipulation_params.brax_vision_ppo_config(env_name)
ppo_params

In [ ]:
x_data, y_data, y_dataerr = [], [], []
times = [datetime.now()]


def progress(num_steps, metrics):
  clear_output(wait=True)

  times.append(datetime.now())
  x_data.append(num_steps)
  y_data.append(metrics["eval/episode_reward"])
  y_dataerr.append(metrics["eval/episode_reward_std"])

  steps = ppo_params["num_timesteps"]
  plt.xlim([steps * -0.1, steps * 1.25])
  plt.ylim([0, 14])
  plt.xlabel("# environment steps")
  plt.ylabel("reward per episode")
  plt.title(f"y={y_data[-1]:.3f}")
  plt.errorbar(x_data, y_data, yerr=y_dataerr, color="blue")

  display(plt.gcf())


ppo_training_params = dict(ppo_params)
network_factory = ppo_networks_vision.make_ppo_networks_vision
if "network_factory" in ppo_params:
  del ppo_training_params["network_factory"]
  network_factory = functools.partial(
      ppo_networks_vision.make_ppo_networks_vision,
      **ppo_params.network_factory
  )

num_eval_envs = ppo_params.get("num_eval_envs", 128)

if "num_eval_envs" in ppo_training_params:
  del ppo_training_params["num_eval_envs"]

train_fn = functools.partial(
    ppo.train,
    **dict(ppo_training_params),
    network_factory=network_factory,
    progress_fn=progress,
    seed=1,
    num_eval_envs=num_eval_envs
)

In [ ]:
eval_env_cfg = config_dict.ConfigDict(env_cfg)
eval_env_cfg.vision_config.nworld = num_eval_envs
eval_env = registry.load(env_name, config=eval_env_cfg)

make_inference_fn, params, metrics = train_fn(
    environment=wrapper.wrap_for_brax_training(env, episode_length=env_cfg.episode_length, action_repeat=1),
    wrap_env=False,
    eval_env=wrapper.wrap_for_brax_training(
        eval_env,
        episode_length=eval_env_cfg.episode_length,
        action_repeat=1
    ),
)
print(f"time to jit: {times[1] - times[0]}")
print(f"time to train: {times[-1] - times[1]}")

## Visualize Rollouts

In [ ]:
infer_env_cfg = config_dict.ConfigDict(env_cfg)
infer_env_cfg.vision_config.nworld = 64
infer_env = registry.load(env_name, config=infer_env_cfg)
wrapped_infer_env = wrapper.wrap_for_brax_training(
    infer_env,
    episode_length=ppo_params.episode_length,
    action_repeat=ppo_params.get("action_repeat", 1),
)

jit_reset = jax.jit(wrapped_infer_env.reset)
jit_step = jax.jit(wrapped_infer_env.step)
jit_inference_fn = jax.jit(make_inference_fn(params, deterministic=True))

In [ ]:
rng = jax.random.split(jax.random.PRNGKey(42), 64)
reset_states = jax.jit(wrapped_infer_env.reset)(rng)

empty_data = reset_states.data.__class__(
    **{k: None for k in reset_states.data.__annotations__}
)  # pytype: disable=attribute-error
empty_traj = reset_states.__class__(
    **{k: None for k in reset_states.__annotations__}
)  # pytype: disable=attribute-error
empty_traj = empty_traj.replace(data=empty_data)

def step(carry, _):
  state, rng = carry
  rng, act_key = jax.random.split(rng)
  act_keys = jax.random.split(act_key, 64)
  act = jax.vmap(jit_inference_fn)(state.obs, act_keys)[0]
  state = wrapped_infer_env.step(state, act)
  traj_data = empty_traj.tree_replace({
      "data.qpos": state.data.qpos,
      "data.qvel": state.data.qvel,
      "data.time": state.data.time,
      "data.ctrl": state.data.ctrl,
      "data.mocap_pos": state.data.mocap_pos,
      "data.mocap_quat": state.data.mocap_quat,
      "data.xfrc_applied": state.data.xfrc_applied,
  })
  return (state, rng), traj_data

@jax.jit
def do_rollout(state, rng):
  _, traj = jax.lax.scan(
      step, (state, rng), None, length=ppo_params.episode_length
  )
  return traj

traj_stacked = do_rollout(reset_states, jax.random.PRNGKey(42 + 1))
# traj_stacked has shape (time, nworld, ...), swap to (nworld, time, ...).
traj_stacked = jax.tree.map(lambda x: jp.moveaxis(x, 0, 1), traj_stacked)

# Use only world 0 and render a single video.
traj_world0 = jax.tree.map(lambda x: x[0], traj_stacked)
rollout = [
    jax.tree.map(lambda x, j=j: x[j], traj_world0)
    for j in range(ppo_params.episode_length)
]

render_every = 2
fps = 1.0 / infer_env.dt / render_every
print(f"FPS for rendering: {fps}")
scene_option = mujoco.MjvOption()
scene_option.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT] = False
scene_option.flags[mujoco.mjtVisFlag.mjVIS_PERTFORCE] = False
scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTFORCE] = False

traj = rollout[::render_every]
frames = infer_env.render(
    traj, height=480, width=640, scene_option=scene_option
)
media.show_video(frames, fps=fps)

## Visualize Domain Randomization

We can visualize the domain randomization by rendering a few environments. We can do this by wrapping the environment manually with our domain randomization function, as opposed to letting brax wrap it for us as we did during training.

In [ ]:
from mujoco_playground._src.manipulation.franka_emika_panda import randomize_vision as randomize

env_cfg.vision_config.nworld = 1024
rand_env = registry.load(env_name, config=env_cfg)

randomization_fn = functools.partial(randomize.domain_randomize,
                                        num_worlds=1024
)

wrapped_env = wrapper.wrap_for_brax_training(
    rand_env,
    episode_length=200,
    action_repeat=1,
    randomization_fn=randomization_fn,
)

jit_reset = jax.jit(wrapped_env.reset)
jit_step = jax.jit(wrapped_env.step)

def tile(img, d):
    assert img.shape[0] == d*d
    img = img.reshape((d,d)+img.shape[1:])
    return np.concat(np.concat(img, axis=1), axis=1)

state = jit_reset(jax.random.split(jax.random.PRNGKey(0), 1024))
media.show_image(tile(state.obs['pixels/view_0'][:64], 8), width=512)

🙌 Thanks for stopping by The Playground!